In [1]:
print("Runnin the notebook")

Runnin the notebook


# LangGraph Tool Calling

* `@tool` → **Converts a Python function into a LangChain Tool**, so the LLM can identify and request it.
* `bind_tools(tools)` → **Connects the tools to the LLM** by giving it their names, descriptions, and input schemas.
* `llm.invoke()` → **Sends the conversation to the LLM**; the LLM decides whether to answer directly or request a tool.
* `ToolNode(tools)` → **Executes the tool call** requested by the LLM and returns the tool's result.
* `tools_condition` → **Routes the graph** based on the LLM response: tool call → `ToolNode`, no tool call → `END`.
* `tools → chat_node` → **Sends the tool result back to the LLM**, allowing it to interpret the result and generate the final answer.
* `.invoke()` → **Actually runs** a LangChain Runnable or Tool with the required input.

### Remember

**`@tool` = Create → `bind_tools` = Connect → LLM = Decide → `ToolNode` = Execute → LLM = Answer**


In [5]:
import os
import random
import requests
from typing import Dict, Any, List, TypedDict, Annotated

from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool

from langchain_groq import ChatGroq

from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

True

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    groq_api_key= os.getenv("GROQ_API_KEY")
)

In [ ]:
search_tool = DuckDuckGoSearchRun(region="us-en")
alpha_key = os.getenv("ALPHA_VANTAGE_KEY")

@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict:
    """
        Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
        using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={alpha_key}"
    r = requests.get(url)
    return r.json()
# get_stock_price.invoke({"symbol": "TSLA"})

1. @tool converts my Python function into a LangChain Tool object. Therefore, instead of calling it like a normal Python function, I use .invoke() with structured arguments. This also allows an LLM agent to discover and call the tool consistently.

2. "`requests` is a third-party Python library used to communicate with REST APIs over HTTP. In my code, `requests.get()` sends a GET request to Alpha Vantage and `response.json()` converts the JSON response into a Python object."


In [ ]:
tools= [get_stock_price,search_tool, calculator]

llm_with_tools = llm.bind_tools(tools)

{'Global Quote': {'01. symbol': 'TSLA',
  '02. open': '357.0700',
  '03. high': '358.8000',
  '04. low': '345.2000',
  '05. price': '348.7500',
  '06. volume': '32864721',
  '07. latest trading day': '2026-08-28',
  '08. previous close': '354.8100',
  '09. change': '-6.0600',
  '10. change percent': '-1.7080%'}}